# 01 — Dataset Inventory

Every dataset fetched in Phase 1, with provenance, size, and licence, read directly
from `data/manifests/*.json`. This notebook is the audit trail for "what did we actually
get" before any analysis runs on top of it.

In [1]:
import sys
sys.path.insert(0, "..")
import json
from pathlib import Path
import pandas as pd

ROOT = Path("..").resolve()
manifests = sorted((ROOT / "data" / "manifests").glob("*.json"))
rows = []
for path in manifests:
    m = json.loads(path.read_text())
    rows.append({
        "dataset": m["dataset"],
        "source_org": m["source_org"],
        "license": m["license"],
        "independence_group": m.get("independence_group"),
        "claim_type": m.get("claim_type"),
        "files": len(m.get("files", [])),
        "MB": round(sum(f["bytes"] for f in m.get("files", [])) / 1e6, 1),
    })
df = pd.DataFrame(rows).sort_values("dataset").reset_index(drop=True)
df

,dataset,source_org,license,independence_group,claim_type,files,MB
0,HMA_DEM8m_MOS,NASA NSIDC,public domain (NASA),NaN,NaN,2,725.5
1,chirps,"UCSB Climate Hazards Center, CHIRPS 2.0",public domain (UCSB Climate Hazards Center),chirps_precipitation,observation,82,0.1
2,chirps_daily_prelim,"UCSB Climate Hazards Center, CHIRPS 2.0",public domain (UCSB Climate Hazards Center),chirps_precipitation,observation,31,0.0
3,hmaglofdb,"Shrestha, Steiner et al. 2023, ESSD 15:3941",CC-BY-4.0,hmaglofdb_record,observation,2,0.5
4,hot_flood_npl,Humanitarian OpenStreetMap Team (HOT),Open Database License (ODC-ODbL),hot_osm_mapping,observation,20,1.8
5,hot_flood_npl_buildings_damage,Humanitarian OpenStreetMap Team (HOT),Creative Commons Attribution International (CC...,cv_damage_vhr,model_output,2,0.4
6,icimod_glacial_lakes,ICIMOD / UNDP,"ICIMOD RDS data download agreement, research use",icimod_inventory,observation,16,3.8
7,opera_l3_dist_alert_hls_v1,NASA / JPL OPERA,public domain (NASA),opera_optical_disturbance,observation,168,93.1
8,opera_l3_dswx_s1_v1,NASA / JPL OPERA,public domain (NASA),opera_radar_water,observation,90,98.0
9,sentinel_1_rtc,Microsoft Planetary Computer / ESA Copernicus,CC-BY-4.0 (Copernicus),sanket_radar,observation,21,1707.0


## Total bronze volume and licence stack

In [2]:
print(f"total files: {df['files'].sum()}")
print(f"total volume: {df['MB'].sum()/1024:.2f} GB")
print()
print("licence stack:")
print(df.groupby('license')['dataset'].apply(list).to_string())

total files: 611
total volume: 3.81 GB

licence stack:
license
CC-BY-4.0                                                                         [hmaglofdb, worldpop]
CC-BY-4.0 (Copernicus)                                                 [sentinel_1_rtc, sentinel_2_l2a]
Creative Commons Attribution International (CC BY)                     [hot_flood_npl_buildings_damage]
ICIMOD RDS data download agreement, research use                                 [icimod_glacial_lakes]
Open Database License (ODC-ODbL)                                                        [hot_flood_npl]
public domain (NASA)                                  [HMA_DEM8m_MOS, opera_l3_dist_alert_hls_v1, op...
public domain (UCSB Climate Hazards Center)                               [chirps, chirps_daily_prelim]


## The ICIMOD PDGL inventory — Scout's population

47 potentially dangerous glacial lakes, exactly matching the ICIMOD/UNDP 2020 figure
quoted throughout the product spec.

In [3]:
from core.connectors.icimod import read_pdgl, read_inventory, source_catchment_gap

pdgl = read_pdgl()
print(f"PDGL count: {len(pdgl)}")
print(pdgl.Country.value_counts())
print()
print("by basin:")
print(pdgl.Basin.value_counts())

PDGL count: 47
Country
China    25
Nepal    21
India     1
Name: count, dtype: int64

by basin:
Basin
Koshi      42
Gandaki     3
Karnali     2
Name: count, dtype: int64


## The gap that motivated this project

Every one of the 47 PDGLs sits above a river corridor. But the lake that formed the
2026 barrier lake source was never on that list — check the source catchment
(the upper Lhende, in China) against the wider 2015 inventory.

In [4]:
gap = source_catchment_gap((85.25, 28.15, 85.55, 28.45))
gap

{'lakes_in_catchment': 25,
 'pdgl_listed': 0,
 'in_china': 23,
 'median_area_km2': 0.0127674880531,
 'max_area_km2': 0.108650225587}

**25 catalogued glacial lakes sit in the source catchment. Zero are PDGL-listed.**
Median area 0.013 km² — below anything a 47-lake priority list would ever flag.
This is the empirical version of the product spec's claim that "the gap was never
data. Nobody was looking, continuously, at all of it."

## HMAGLOFDB — the historical base rate

In [5]:
import pandas as pd
glof = pd.read_csv(ROOT / "data" / "bronze" / "hmaglofdb" / "HMAGLOFDB.csv",
                    low_memory=False, encoding="latin-1")
print(f"total recorded GLOFs: {len(glof)}")
print(glof.Country.value_counts().head(8))
print()
print("lake type:")
print(glof.Lake_type.value_counts())
print()
nepal = glof[glof.Country == "Nepal"]
print(f"Nepal events: {len(nepal)}, total lives lost where recorded: "
      f"{pd.to_numeric(nepal.Lives_total, errors="coerce").fillna(0).sum():.0f}")

total recorded GLOFs: 773
Country
China         205
Kyrgyzstan    198
Pakistan      151
India          61
Nepal          58
Kazakhstan     54
Bhutan         21
Tajikistan     20
Name: count, dtype: int64

lake type:
Lake_type
Moraine dammed      387
Ice dammed          237
Supraglacial         85
Unknown              30
Water pocket         22
Bedrock               6
Landslide dammed      3
Thermokarst           3
Name: count, dtype: int64

Nepal events: 58, total lives lost where recorded: 36


## USGS ANSS — the reclassification, confirmed live

The product spec states USGS reclassified the 26 August 2026 seismic signal from an
M4.4 earthquake to an M5.2 landslide-type event. Query the live ANSS catalogue.

In [6]:
from datetime import date
from core.connectors.usgs import events_near

events = events_near(85.377, 28.271, radius_km=80, start=date(2026, 8, 25),
                      end=date(2026, 8, 29), min_magnitude=3.0)
pd.DataFrame(events)

,id,time,mag,type,place
0,us7000tc90,1787724035000,4.2,landslide,"55 km NW of Kodāri̇̄, Nepal"
1,us7000tbwb,1787712730000,5.2,landslide,"55 km NW of Kodāri̇̄, Nepal"


**Confirmed: the ANSS catalogue already carries `type: landslide` for the M5.2 event.**